# 6.1 TextVectorization

這份 Notebook 示範如何使用 Keras `TextVectorization` 將文字轉成 token index、multi-hot、count 與 TF-IDF 特徵，並建立簡單文字分類模型。


## 1. 載入套件


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(42)


## 2. 建立小型文字資料


In [ ]:
def make_sentiment_data(repeats=90, seed=42):
    rng = np.random.default_rng(seed)
    positive_templates = [
        'the product is great and useful',
        'the service is fast and friendly',
        'the tutorial is clear and helpful',
        'the model works well today',
        'the result is excellent and reliable',
        'the app feels smooth and stable',
    ]
    negative_templates = [
        'the product is bad and broken',
        'the service is slow and rude',
        'the tutorial is confusing and unclear',
        'the model fails again today',
        'the result is wrong and unstable',
        'the app feels buggy and slow',
    ]
    suffixes = ['now', 'for me', 'in this test', 'with this update', 'on my device']
    texts, labels = [], []
    for _ in range(repeats):
        for sentence in positive_templates:
            texts.append(sentence + ' ' + rng.choice(suffixes))
            labels.append(1)
        for sentence in negative_templates:
            texts.append(sentence + ' ' + rng.choice(suffixes))
            labels.append(0)
    texts = np.array(texts, dtype=object)
    labels = np.array(labels, dtype='int64')
    idx = rng.permutation(len(labels))
    return texts[idx], labels[idx]

texts, labels = make_sentiment_data(repeats=20, seed=1)
x_train, x_test, y_train, y_test = train_test_split(texts, labels, test_size=0.25, random_state=42, stratify=labels)
pd.DataFrame({'text': x_train[:6], 'label': y_train[:6]})


## 3. token index 序列

`output_mode='int'` 最常用於 Embedding、LSTM、CNN 與 Transformer。


In [ ]:
vectorizer_int = tf.keras.layers.TextVectorization(
    max_tokens=1000,
    output_mode='int',
    output_sequence_length=10,
)
vectorizer_int.adapt(x_train)
print(vectorizer_int.get_vocabulary()[:20])
vectorizer_int(tf.constant(x_train[:3])).numpy()


## 4. multi-hot、count 與 TF-IDF


In [ ]:
for mode in ['multi_hot', 'count', 'tf_idf']:
    layer = tf.keras.layers.TextVectorization(max_tokens=1000, output_mode=mode)
    layer.adapt(x_train)
    output = layer(tf.constant(x_train[:2])).numpy()
    print(mode, output.shape)
    print(output[:1, :10])


## 5. 建立簡單文字分類模型


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(), dtype=tf.string),
    vectorizer_int,
    tf.keras.layers.Embedding(input_dim=1000, output_dim=16),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, validation_split=0.2, epochs=12, batch_size=16, verbose=0)
model.evaluate(x_test, y_test, verbose=0, return_dict=True)


## 6. 預測新句子


In [ ]:
new_texts = np.array(['the tutorial is clear and useful', 'the app is slow and broken'], dtype=object)
prob = model.predict(new_texts, verbose=0).ravel()
pd.DataFrame({'text': new_texts, 'positive_probability': prob})


## 7. 如何套用自己的資料

整理自己的 `texts` 與 `labels`，用訓練文字呼叫 `adapt()`。若後面要接 Embedding，使用 `output_mode='int'`；若只是快速 baseline，可使用 `multi_hot` 或 `tf_idf`。


## 8. 小結

`TextVectorization` 把文字標準化、切詞、vocabulary 與向量化整合成 Keras layer，是後續 NLP 模型的共通入口。
